# Open-Weight Language Models as Constrained Weight-Space Repair Proposers for Frozen Neural Music Dynamics

*Executable research protocol · Academic V2 · Google Colab*

## Abstract

Repairing a trained continuous-time neural model typically requires additional gradient optimization, even when its forecasting errors may be addressable by modifying only a small subset of parameters. We investigate a constrained form of Weight-Space Programming (WSP) in which open-weight language models propose direct edits to a frozen coupled neural PDE–ODE music forecaster. Following training on recording-disjoint log-spectral sequences, the network is frozen and eight entries of a convolutional reaction kernel are designated as the repair surface. A training-only Jacobian of future spectral predictions with respect to these weights is used to summarize local error sensitivity through a Gram matrix and residual projection. Three Hugging Face instruction models—Qwen2.5-0.5B-Instruct, Qwen2.5-1.5B-Instruct, and SmolLM2-1.7B-Instruct—receive these diagnostics and generate bounded numerical weight updates. Each proposal is syntax-checked, applied without modifying other weights, and either accepted or rejected on independent validation recordings. We compare the resulting held-out forecast accuracy with an unchanged network, a Jacobian-regularized least-squares repair, and Adam optimization over the identical eight weights. Outcomes include proposal validity and acceptance, spectral and high-frequency prediction error, onset-proxy accuracy, and computational cost. The central hypothesis is that language-model-generated interventions can produce useful sparse repairs; superiority over numerical or gradient methods is not presumed. The study evaluates constrained kernel editing rather than general-purpose synthesis of arbitrary neural weights.

## 1. Introduction and research objective

Weight-Space Programming [2] suggests that model behavior can sometimes be modified through direct parameter intervention rather than full retraining. Whether such proposals remain reliable for differentiable continuous-time systems is unresolved. Coupled spectral PDE–ODE models [1] provide a useful setting because held-out forecast errors can be measured after a precisely controlled intervention.

**Research question (RQ4).** Can open-weight instruction models propose bounded updates to *actual frozen neural convolution weights* that improve recording-disjoint music forecasts, and do these proposals offer any advantage relative to Jacobian-guided numerical repair or Adam on the same eight-dimensional edit surface?

**Contribution under test.** This is an implementation and comparative evaluation of constrained WSP applied to a frozen continuous-time music model. The design introduces an independent validation gate and explicitly records invalid, rejected, and unavailable model proposals. It is not a claim to reproduce the original WSP experiments or to edit the entire network.

### 1.1. Unresolved problem: when can reasoning-driven weight editing outperform optimization?

Weight-Space Programming [2] suggests that an LLM can sometimes identify and modify model mechanisms through their numerical weights. Existing demonstrations are selective and do not show that arbitrary edits to a stiff neural differential equation will preserve dynamical stability. A fixed eight-entry kernel edit is a tractable probe, but is unlikely to repair failures rooted in representation capacity, unobserved latent dynamics, or dataset shift.

**Critical question CQ6 — mechanism-localized, non-regressive model repair.** Can an open-weight reasoning model select and modify a *structurally responsible* neural parameter subset, producing improvements on shifted audio without degrading performance on the original recording distribution?

**Research direction.** Define two disjoint validation domains (original and shifted) and require candidate edits to satisfy a non-regression constraint. Let the controller localize a parameter subspace using sensitivity and residual evidence, propose weight edits with three independent Hugging Face models, and verify both predictive gain and numerical stability. The current notebook **proposes edits to eight predetermined weights and validates one objective**. Automatic localization, two-domain non-regression, and robustness guarantees are proposed future experiments, not implemented claims.

**Critical question CQ7 — when is an LLM an optimizer rather than an expensive interface to least squares?** Because the prompt contains the Jacobian Gram matrix and residual projection, the LLM can imitate a straightforward numerical update. A meaningful positive result requires outperforming direct Jacobian repair, Adam, low-rank updates, random bounded edits, and full fine-tuning under comparable compute budgets, especially when a structural edit is needed.

**Falsifiable criterion.** Measure accepted-repair probability, new-domain MSE, original-domain degradation, number of edited weights, solver divergence, wall-clock/GPU cost, and invalid JSON rate across at least ten controlled seeds. Record rejected proposals and failures; never replace them with numerical candidates. No published advantage of this constrained PCWSR variant is presumed.

## 2. Proposed method: agentic propose–verify weight-space intervention

Let $f_\theta$ be a trained coupled neural PDE–ODE forecaster and $S$ a set of eight selected entries of its final convolutional reaction kernel. The frozen network is repaired only by a bounded vector $\delta\in[-\epsilon,\epsilon]^{8}$:

$$
\theta'=\theta+P_S^\top\delta,\qquad
J=\frac{\partial\,\operatorname{vec}f_\theta(X_{\mathrm{train}})}{\partial\delta},\quad
r=\operatorname{vec}(Y_{\mathrm{train}}-f_\theta(X_{\mathrm{train}})).
$$

$P_S$ maps a proposed edit to the selected weight coordinates. The diagnostics $J^\top J$ and $J^\top r$ are computed **only from training recordings** and supplied with the frozen selected weights to each pretrained instruction model. The model outputs a JSON array of eight bounded numbers. A parser rejects malformed, nonfinite, or out-of-range values; accepted candidates are integrated through the frozen model and must reduce *validation* spectral MSE by the specified threshold. Only afterward are final held-out test metrics computed. An unsuccessful proposal causes rollback to the frozen model.

**Comparison arms.** A damped Jacobian least-squares update solves $(J^\top J+\lambda I)\delta=J^\top r$; Adam optimizes the same eight coordinates using training observations. The evaluation therefore tests the **proposal mechanism**, not greater editable parameter capacity. The three Hugging Face models are independent candidate generators; no reported score is substituted if a model download or proposal fails.

## 3. Expected outcomes, hypotheses, and decision criteria

**H1 (repair feasibility).** At least one validated open-weight model proposal improves held-out forecast error relative to the frozen network. **H0:** none of the bounded proposals improves held-out performance under the preregistered protocol. **H2 (relative efficiency).** LLM-generated updates may or may not outperform numerical Jacobian repair and eight-weight Adam; all available comparison arms must be assessed using equal edit capacity and disclosed inference costs. **H3 (failure robustness).** Invalid syntax, validation rejection, or checkpoint unavailability are expected possibilities and must remain visible in the final results.

These hypotheses are stated *before* model execution. They are not empirical conclusions. A single seed, small hand-selected repair surface, and synthetic fallback cannot establish general superiority or transfer to arbitrary hidden-layer editing.

## 4. Experimental procedure

**Step 1 — Data and baseline.** Prepare recording-level splits, train the persistent coupled model, and freeze its full parameter set. **Step 2 — Diagnosis.** Select eight kernel weights; compute training-only Jacobian sensitivity and prediction residual statistics. **Step 3 — Numerical controls.** Fit a regularized Jacobian update and an Adam update over exactly the selected weights. **Step 4 — Agentic proposal.** Query three independently loaded Hugging Face instruction models for bounded JSON edits; record raw outputs, invalidity, and inference time. **Step 5 — Verification.** Apply valid edits to the frozen network, evaluate disjoint validation recordings, and accept only improvements beyond the fixed threshold. **Step 6 — Final evaluation.** Compute test spectral MSE, high-frequency MSE, onset-proxy F1, acceptance status, edit count, and compute cost. The method diagram highlights the propose–verify mechanism rather than the frozen predictor alone.


### Three method diagrams

The architecture graph highlights the **Hugging Face weight proposer** rather than the frozen PDE–ODE alone. The evaluation graph places it alongside numerical Jacobian repair, Adam repair, and full-training baselines. The validation graph shows the explicit reject/rollback boundary.

### Interpretation of the methodological figures

**Figure 1 — Mechanistic architecture:** the highlighted node identifies the *proposed scientific mechanism*, not a generic preprocessing step. The incoming arrows specify evidence or state needed for the method; outgoing arrows specify the predictions or edits being evaluated.

**Figure 2 — Comparative evaluation:** proposed-method results are distinguished from baselines using a thick outline or hatching. A visually highlighted method is **not** an assumed winner. Its score must be measured on the same held-out target as the baselines.

**Figure 3 — Data partitioning and verification:** training diagnostics may inform candidate generation; validation decides acceptance; the test partition is reserved for final reporting. Failed model downloads, invalid outputs, and rejected repairs remain visible and are not replaced by fabricated scores.

These diagrams are executable Graphviz figures generated inside Colab; the paper-style explanations and executable experiments remain the primary deliverable.


In [ ]:
%pip -q install graphviz torchdiffeq scipy pandas matplotlib

In [ ]:
import os, math, time, random, json
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch
from torch import nn
from torchdiffeq import odeint
from scipy.signal import stft, resample_poly
from scipy.io import wavfile
SEED=211100343
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:',device,'torch:',torch.__version__)


In [ ]:
# Three executable research diagrams, saved inside the Colab runtime.
# The thicker proposed-method border is intentionally distinct from baseline methods.
from IPython.display import display, SVG
from graphviz import Digraph

def research_diagram(name, nodes, edges, proposed, caption):
    dot=Digraph(name,format='svg')
    dot.attr(rankdir='LR',fontname='Helvetica',labelloc='t',label=caption,nodesep='0.35',ranksep='0.45')
    for key,label in nodes:
        if key in proposed:
            dot.node(key,label,shape='box',style='rounded,bold',color='#B45309',penwidth='3.5',
                     fillcolor='#FFF7ED',fontcolor='#7C2D12')
        else:
            dot.node(key,label,shape='box',style='rounded',color='#64748B')
    for u,v in edges: dot.edge(u,v)
    try:
        svg=dot.pipe(format='svg').decode('utf-8')
        display(SVG(svg));Path(name+'.svg').write_text(svg)
    except Exception as exc:
        print('Graphviz rendering unavailable:',exc,'\n',dot.source)

research_diagram('v2_method',
    [('audio','Audio / spectrogram'),('diagnose','Diagnostics / local errors'),
     ('local','Local PDE-inspired operator'),('global','Persistent or short-horizon ODE'),
     ('proposer','PROPOSED: model / weight proposer'),('solve','Differentiable integration'),
     ('verify','Independent verifier')],
    [('audio','diagnose'),('diagnose','local'),('diagnose','global'),
     ('local','proposer'),('global','proposer'),('proposer','solve'),('solve','verify')],
    ['local','global','proposer','solve'], 'METHOD: '+'Frozen-model weight-space intervention')
research_diagram('v2_evaluation',
    [('data','Recordings / synthetic signals'),('split','Train → validation → test'),
     ('proposed','PROPOSED: HF neural-weight proposer'),('base','Classical / neural baselines'),
     ('hf','3 Hugging Face models (where applicable)'),('gate','Validation / selection gate'),
     ('out','Held-out MSE, onsets, latency')],
    [('data','split'),('split','proposed'),('split','base'),('split','hf'),
     ('proposed','gate'),('base','gate'),('hf','gate'),('gate','out')],
    ['proposed'], 'EVALUATION: isolate the proposed method from benchmarks')
research_diagram('v2_decisions',
    [('frozen','Frozen candidate / fixed design'),('probe','Residual / sensitivity probe'),
     ('action','PROPOSED: targeted intervention'),('validation','Disjoint validation'),
     ('accepted','Accept only if improvement'),('rejected','Reject / record failure'),
     ('test','One-shot held-out evaluation')],
    [('frozen','probe'),('probe','action'),('action','validation'),
     ('validation','accepted'),('validation','rejected'),('accepted','test'),('rejected','test')],
    ['action'], 'VERIFICATION: no candidate accesses test labels during selection')

## Step 1 — Establish recording-disjoint music observations

The agent either loads an eligible ZIP of WAV tracks or generates varied synthetic recordings. No test recording enters the training or validation set. Standardisation uses training recordings.

In [ ]:
# Configuration and optional real-recording ZIP upload.
import zipfile, tempfile, os, glob, random, gc
from scipy.signal import stft, resample_poly
from scipy.io import wavfile
from torch.utils.data import TensorDataset, DataLoader
BENCH_SEEDS=[11]   # change to >=10 seeds for a paper
BENCH_H=8
BENCH_CONTEXT=8
BENCH_EPOCHS=8
BENCH_BATCH=24
BENCH_BANDS=48
BENCH_SR=8000
BENCH_HOP=128
BENCH_NFFT=512
BENCH_MAX_RECORDINGS=36
BENCH_UPLOAD_ZIP=False
BENCH_ROOT=Path('/content/music_benchmark')
BENCH_ROOT.mkdir(parents=True,exist_ok=True)
if BENCH_UPLOAD_ZIP:
    from google.colab import files
    z=files.upload()
    zipname=next(iter(z))
    with zipfile.ZipFile(zipname) as archive:
        for item in archive.infolist():
            destination=(BENCH_ROOT/item.filename).resolve()
            if not destination.is_relative_to(BENCH_ROOT.resolve()):
                raise ValueError('Unsafe ZIP member path')
            if item.is_dir(): continue
            if not item.filename.lower().endswith('.wav'):continue
            destination.parent.mkdir(parents=True,exist_ok=True)
            destination.write_bytes(archive.read(item))
print('Benchmark configuration:',BENCH_H,'steps,',len(BENCH_SEEDS),'seeds')


In [ ]:
def bench_synth(seed, instrument, tempo, seconds=5.0):
    rng=np.random.default_rng(seed)
    tt=np.arange(int(seconds*BENCH_SR))/BENCH_SR
    x=np.zeros_like(tt)
    base=rng.choice([110.,146.83,164.81,196.,220.,246.94,261.63])
    harmonic_profiles={
      'soft':[1,.15,.05,.02,.01],
      'bright':[1,.7,.45,.25,.13],
      'reed':[1,.8,.12,.5,.07],
      'plucked':[1,.45,.22,.1,.05],
    }
    harms=harmonic_profiles[instrument]
    beat=60/tempo
    for k,start in enumerate(np.arange(0,seconds,beat)):
        note=base*(2**(int(rng.choice([-5,-3,0,2,4,7]))/12))
        length=min(int(beat*BENCH_SR),len(tt)-int(start*BENCH_SR))
        if length<=0:continue
        local=np.arange(length)/BENCH_SR
        decay=3.2 if instrument=='plucked' else .7
        env=np.minimum(local*30,1)*np.exp(-decay*local)
        s=sum(amp*np.sin(2*np.pi*(h+1)*note*local)
              for h,amp in enumerate(harms))
        x[int(start*BENCH_SR):int(start*BENCH_SR)+length]+=env*s
    x+=.003*rng.standard_normal(len(x))
    return (x/(np.max(np.abs(x))+1e-8)).astype(np.float32)

def bench_wav(path):
    sr,a=wavfile.read(path)
    a=a.astype(np.float32)
    if a.ndim>1:a=a.mean(axis=1)
    a/=max(float(np.max(np.abs(a))),1e-8)
    if sr!=BENCH_SR:
        from math import gcd
        g=gcd(int(sr),BENCH_SR)
        a=resample_poly(a,BENCH_SR//g,int(sr)//g)
    return a[:6*BENCH_SR]

def bench_features(a):
    _,_,z=stft(a,fs=BENCH_SR,nperseg=BENCH_NFFT,
               noverlap=BENCH_NFFT-BENCH_HOP,boundary=None)
    mag=np.abs(z).astype(np.float32)
    # Equal-frequency aggregation of the first 240 non-DC bins.
    mag=mag[1:241].reshape(BENCH_BANDS,5,-1).mean(axis=1)
    return np.log1p(20*mag).T.astype(np.float32)

paths=sorted(BENCH_ROOT.rglob('*.wav'))[:BENCH_MAX_RECORDINGS]
recordings=[]
if paths:
    for path in paths:
        try:
            audio=bench_wav(path)
            if len(audio)<2*BENCH_SR:continue
            recordings.append({'id':str(path.relative_to(BENCH_ROOT)),
                               'label':path.parent.name,
                               'features':bench_features(audio)})
        except Exception as e:
            print('Skipping',path.name,str(e)[:100])
else:
    for i in range(30):
        instrument=['soft','bright','reed','plucked'][i%4]
        tempo=[72,92,118,145,168][(i//4)%5]
        recordings.append({'id':f'synth_{i:02d}','label':instrument,
                           'tempo':tempo,
                           'features':bench_features(bench_synth(i+200,instrument,tempo))})
print('Recording count:',len(recordings),'labels:',sorted(set(r['label'] for r in recordings)))
assert len(recordings)>=9,'Need >=9 recordings for train/val/test split'


In [ ]:
# Deterministic recording-level split and training-only normalization.
def bench_split(seed):
    rng=np.random.default_rng(seed)
    # split each label to avoid losing instrument categories when feasible
    groups={}
    for r in recordings:groups.setdefault(r['label'],[]).append(r)
    train,val,test=[],[],[]
    for label,items in sorted(groups.items()):
        items=list(items);rng.shuffle(items)
        if len(items)>=3:
            nv=max(1,round(.2*len(items)));nt=max(1,round(.2*len(items)))
            val.extend(items[:nv]);test.extend(items[nv:nv+nt]);train.extend(items[nv+nt:])
        else:train.extend(items)
    if not val or not test:raise ValueError('Need >=3 recordings per label for stratified split')
    assert not ({r['id'] for r in train}&{r['id'] for r in test})
    return train,val,test

def bench_prepare(seed):
    train,val,test=bench_split(seed)
    tr=np.concatenate([r['features'] for r in train])
    mean=tr.mean(axis=0);std=tr.std(axis=0)+.01
    def make(items):
        xx,yy,ids=[],[],[]
        for r in items:
            a=np.clip((r['features']-mean)/std,-6,6)
            for start in range(0,len(a)-BENCH_CONTEXT-BENCH_H+1,6):
                xx.append(a[start:start+BENCH_CONTEXT])
                yy.append(a[start+BENCH_CONTEXT:start+BENCH_CONTEXT+BENCH_H])
                ids.append(r['id'])
        return torch.tensor(np.stack(xx),dtype=torch.float32),torch.tensor(np.stack(yy),dtype=torch.float32),ids
    return make(train),make(val),make(test),{'train':[r['id'] for r in train],
      'validation':[r['id'] for r in val],'test':[r['id'] for r in test]}


## Step 2 — Instantiate the persistent coupled spectral dynamics

We integrate spectral state and global state through the full horizon. The target of repair is a real, frozen Conv1d kernel, not a decorative adapter.

In [ ]:
import torch.nn.functional as F
class PersistentCoupled(nn.Module):
    def __init__(self, mode='coupled', bands=BENCH_BANDS, latent=12):
        super().__init__()
        self.mode=mode
        self.bands=bands;self.latent=latent
        self.history=nn.GRU(bands,latent,batch_first=True)
        self.local=nn.Sequential(nn.Conv1d(1,12,5,padding=2),nn.Tanh(),
                                 nn.Conv1d(12,1,3,padding=1))
        self.z_dynamics=nn.Sequential(nn.Linear(latent+1,24),nn.Tanh(),
                                      nn.Linear(24,latent))
        self.coupling=nn.Linear(latent,bands,bias=False)
        self.diffusion_raw=nn.Parameter(torch.tensor(-2.))
    def forward(self,t,state):
        u,z=state
        padded=F.pad(u[:,None,:],(1,1),mode='replicate')[:,0]
        lap=padded[:,:-2]-2*u+padded[:,2:]
        D=F.softplus(self.diffusion_raw)
        if self.mode=='no_diffusion':D=D*0
        du=self.local(u[:,None,:])[:,0]+self.coupling(z)+D*lap
        dz=self.z_dynamics(torch.cat([z,u.mean(-1,keepdim=True)],dim=-1))
        return du,dz
    def predict(self,x,dt,method='rk4',rtol=1e-4,atol=1e-5):
        _,z=self.history(x);z=z[-1]
        u=x[:,-1]
        t=torch.arange(BENCH_H+1,device=x.device,dtype=x.dtype)*dt
        opts={'step_size':dt/2} if method=='rk4' else None
        states=odeint(self,(u,z),t,method=method,options=opts,rtol=rtol,atol=atol)
        return states[0][1:].transpose(0,1)

class ConventionalNeuralODE(nn.Module):
    def __init__(self,bands=BENCH_BANDS,hidden=48):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(bands,hidden),nn.Tanh(),
                               nn.Linear(hidden,bands))
    def forward(self,t,u):return self.net(u)
    def predict(self,x,dt,method='rk4',rtol=1e-4,atol=1e-5):
        t=torch.arange(BENCH_H+1,device=x.device,dtype=x.dtype)*dt
        opts={'step_size':dt/2} if method=='rk4' else None
        return odeint(self,x[:,-1],t,method=method,options=opts,rtol=rtol,atol=atol)[1:].transpose(0,1)

class TemporalCNN(nn.Module):
    def __init__(self,bands=BENCH_BANDS,hidden=64):
        super().__init__()
        self.net=nn.Sequential(nn.Conv1d(bands,hidden,3,padding=1),nn.ReLU(),
                               nn.Conv1d(hidden,hidden,3,padding=1),nn.ReLU(),
                               nn.AdaptiveAvgPool1d(1))
        self.head=nn.Linear(hidden,BENCH_H*bands)
    def predict(self,x,dt,**kw):
        return self.head(self.net(x.transpose(1,2)).squeeze(-1)).reshape(-1,BENCH_H,x.shape[-1])

class GRUForecast(nn.Module):
    def __init__(self,bands=BENCH_BANDS,hidden=64):
        super().__init__()
        self.gru=nn.GRU(bands,hidden,batch_first=True)
        self.head=nn.Linear(hidden,BENCH_H*bands)
    def predict(self,x,dt,**kw):
        _,h=self.gru(x)
        return self.head(h[-1]).reshape(-1,BENCH_H,x.shape[-1])

class TinyTransformer(nn.Module):
    def __init__(self,bands=BENCH_BANDS,d_model=48):
        super().__init__()
        self.proj=nn.Linear(bands,d_model)
        self.pos=nn.Parameter(torch.zeros(1,BENCH_CONTEXT,d_model))
        layer=nn.TransformerEncoderLayer(d_model,4,dim_feedforward=96,
                 dropout=0,batch_first=True)
        self.encoder=nn.TransformerEncoder(layer,num_layers=1)
        self.head=nn.Linear(d_model,BENCH_H*bands)
    def predict(self,x,dt,**kw):
        h=self.encoder(self.proj(x)+self.pos)
        return self.head(h[:,-1]).reshape(-1,BENCH_H,x.shape[-1])

BENCH_MODELS={
 'coupled':lambda:PersistentCoupled('coupled'),
 'no_diffusion':lambda:PersistentCoupled('no_diffusion'),
 'neural_ode':ConventionalNeuralODE,
 'tcn':TemporalCNN,'gru':GRUForecast,'transformer':TinyTransformer,
}
BENCH_DT=BENCH_HOP/BENCH_SR
print('Parameters by architecture:')
for name,ctor in BENCH_MODELS.items():
    model=ctor()
    print(name,sum(p.numel() for p in model.parameters()))


## Step 3 — Specify physically and musically relevant tests

Spectral prediction MSE is primary. High-frequency error and onset proxy F1 are complementary; they can expose diffusion-induced smoothing.

In [ ]:
def bench_flux(x,context_last):
    previous=torch.cat([context_last[:,None,:],x[:,:-1]],dim=1)
    return F.relu(x-previous).sum(-1)
def bench_onset_metrics(pred,true,context_last,threshold):
    pf=bench_flux(pred,context_last).cpu().numpy()
    tf=bench_flux(true,context_last).cpu().numpy()
    tp=fp=fn=0
    offsets=[]
    for p,t in zip(pf,tf):
        pidx=np.where(p>threshold)[0].tolist()
        tidx=np.where(t>threshold)[0].tolist()
        used=set()
        for pi in pidx:
            eligible=[ti for ti in tidx if ti not in used and abs(ti-pi)<=1]
            if eligible:
                ti=min(eligible,key=lambda k:abs(k-pi))
                used.add(ti);tp+=1;offsets.append(abs(ti-pi))
            else:fp+=1
        fn+=len(tidx)-len(used)
    precision=tp/max(tp+fp,1);recall=tp/max(tp+fn,1)
    f1=2*precision*recall/max(precision+recall,1e-12)
    return {'onset_f1':f1,
            'transient_localization_mae_frames':float(np.mean(offsets)) if offsets else None,
            'reference_onset_events':tp+fn,'predicted_onset_events':tp+fp}
def bench_evaluate(pred,y,x,threshold):
    result={'mse':float((pred-y).square().mean()),
            'hf_mse':float((pred[:,:,-BENCH_BANDS//4:]-y[:,:,-BENCH_BANDS//4:]).square().mean()),
            'per_horizon_mse':(pred-y).square().mean((0,2)).cpu().tolist()}
    result.update(bench_onset_metrics(pred,y,x[:,-1],threshold))
    return result


In [ ]:
def bench_fit_one(name,ctor,train,val,seed):
    torch.manual_seed(seed)
    model=ctor().to(device)
    optimizer=torch.optim.Adam(model.parameters(),lr=1e-3)
    train_x,train_y,_=train
    val_x,val_y,_=val
    loader=DataLoader(TensorDataset(train_x,train_y),batch_size=BENCH_BATCH,shuffle=True)
    best=float('inf');checkpoint=None
    if device.type=='cuda':torch.cuda.reset_peak_memory_stats()
    start=time.perf_counter()
    for epoch in range(BENCH_EPOCHS):
        model.train()
        for bx,by in loader:
            bx=bx.to(device);by=by.to(device)
            optimizer.zero_grad()
            pred=model.predict(bx,BENCH_DT)
            loss=(pred-by).square().mean()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.)
            optimizer.step()
        model.eval()
        with torch.no_grad():
            vloss=[]
            for bx,by in DataLoader(TensorDataset(val_x,val_y),batch_size=BENCH_BATCH*2):
                vloss.append(float((model.predict(bx.to(device),BENCH_DT)-by.to(device)).square().mean()))
            v=float(np.mean(vloss))
        if v<best:
            best=v;checkpoint={k:val.detach().cpu().clone() for k,val in model.state_dict().items()}
    train_s=time.perf_counter()-start
    model.load_state_dict(checkpoint)
    model.eval()
    peak=(torch.cuda.max_memory_allocated()/1024**2) if device.type=='cuda' else None
    return model,{'training_s':train_s,'validation_mse':best,
                  'parameters':sum(p.numel() for p in model.parameters()),
                  'peak_gpu_memory_mib':peak}

def bench_predict_batches(model,x):
    predictions=[]
    with torch.no_grad():
        for batch in x.split(BENCH_BATCH*2):
            predictions.append(model.predict(batch.to(device),BENCH_DT).cpu())
    return torch.cat(predictions)

def bench_timed_predictions(model,x):
    if device.type=='cuda':torch.cuda.synchronize()
    start=time.perf_counter()
    pred=bench_predict_batches(model,x)
    if device.type=='cuda':torch.cuda.synchronize()
    return pred,1000*(time.perf_counter()-start)/len(x)



## Step 4 — Train the base model and strong neural baselines

All base weights are frozen after training. The baseline architectures give context for whether a repaired coupled model is competitive at all.

In [ ]:
import copy, gc
WSP_SEED=11
torch.manual_seed(WSP_SEED);np.random.seed(WSP_SEED);random.seed(WSP_SEED)
train,val,test,WSP_SPLIT=bench_prepare(WSP_SEED)
train_x,train_y,_=train;val_x,val_y,_=val;test_x,test_y,_=test
WSP_THRESHOLD=float(torch.quantile(bench_flux(val_y,val_x[:,-1]).flatten(),.85))
WSP_BASE, WSP_BASE_STATS=bench_fit_one('coupled',BENCH_MODELS['coupled'],train,val,WSP_SEED)
for p in WSP_BASE.parameters():p.requires_grad_(False)
WSP_BASE.eval()
WSP_ARCH_RESULTS={}
for name in ('tcn','gru','transformer','neural_ode'):
    model,stats=bench_fit_one(name,BENCH_MODELS[name],train,val,WSP_SEED)
    pred,ms=bench_timed_predictions(model,test_x)
    WSP_ARCH_RESULTS[name]={**stats,**bench_evaluate(pred,test_y,test_x,WSP_THRESHOLD),
                             'inference_ms_per_window':ms}
    del model
    if device.type=='cuda':torch.cuda.empty_cache()
print('Base model trained and frozen. Test recordings:',WSP_SPLIT['test'])


## Step 5 — Localise an eight-weight repair surface

The agent picks eight entries of the final convolution kernel and constructs a training-only Jacobian. This restricts the hypothesis class so that direct numerical, Adam and language-model proposals operate on **identical editable coordinates**.

In [ ]:
from torch.func import functional_call
WSP_PARAM='local.2.weight'
assert WSP_PARAM in dict(WSP_BASE.named_parameters())
WSP_ORIGINAL=dict(WSP_BASE.named_parameters())[WSP_PARAM].detach().clone()
WSP_INDEX=torch.arange(8,device=device)
WSP_FROZEN_VALUES=WSP_ORIGINAL.flatten()[WSP_INDEX].detach().clone()
WSP_EDIT_BOUND=.20
WSP_L2=.02

def wsp_model_prediction(delta,x):
    # functional_call replaces the selected kernel tensor without mutating frozen weights.
    # Functional evaluation is implemented by replacing the parameter in a local model
    # wrapper; `odeint` invokes this modified module throughout the full horizon.
    class EditedModel(nn.Module):
        def __init__(self,base,delta):
            super().__init__()
            self.base=base
            self.delta=delta
        def forward(self,t,state):
            u,z=state
            pad=F.pad(u[:,None,:],(1,1),mode='replicate')[:,0]
            lap=pad[:,:-2]-2*u+pad[:,2:]
            kernel=self.base.local[2].weight.detach().clone().flatten()
            kernel=kernel.scatter(0,WSP_INDEX,WSP_FROZEN_VALUES+self.delta)
            kernel=kernel.view_as(self.base.local[2].weight)
            conv1=self.base.local[0](u[:,None,:])
            act=self.base.local[1](conv1)
            local=F.conv1d(act,kernel,bias=self.base.local[2].bias,padding=1)[:,0]
            du=local+self.base.coupling(z)+F.softplus(self.base.diffusion_raw)*lap
            dz=self.base.z_dynamics(torch.cat([z,u.mean(-1,keepdim=True)],dim=-1))
            return du,dz
        def predict(self,x,dt):
            _,z=self.base.history(x);z=z[-1]
            t=torch.arange(BENCH_H+1,device=x.device,dtype=x.dtype)*dt
            state=odeint(self,(x[:,-1],z),t,method='rk4',options={'step_size':dt/2})
            return state[0][1:].transpose(0,1)
    return EditedModel(WSP_BASE,delta).predict(x,BENCH_DT)

WSP_ZERO=torch.zeros(8,device=device)
def wsp_mse(delta,x,y,batch=24):
    values=[]
    with torch.no_grad():
        for a,b in zip(x.split(batch),y.split(batch)):
            values.append((wsp_model_prediction(delta,a.to(device))-b.to(device)).square().mean().item()*len(a))
    return sum(values)/len(x)
WSP_BASE_VAL=wsp_mse(WSP_ZERO,val_x,val_y)
print('Frozen validation MSE:',WSP_BASE_VAL,'| selected weights:',WSP_FROZEN_VALUES.cpu().tolist())


## Step 6 — Compute direct and Adam repair baselines

The numerical solution uses damped sensitivity regression; the Adam solution trains only the selected weights. Both methods accept a change only if validation MSE improves by at least 0.5%.

In [ ]:
WSP_JAC_BATCH=2
xj=train_x[:WSP_JAC_BATCH].to(device)
yj=train_y[:WSP_JAC_BATCH].to(device)
wsp_start=time.perf_counter()
def flat_prediction(delta):
    return wsp_model_prediction(delta,xj).reshape(-1)
with torch.enable_grad():
    J=torch.autograd.functional.jacobian(flat_prediction,WSP_ZERO,vectorize=False)
    residual=(yj.reshape(-1)-flat_prediction(WSP_ZERO)).detach()
gram=J.T@J
cross=J.T@residual
proposal=torch.linalg.solve(gram+WSP_L2*torch.eye(8,device=device),cross).detach()
WSP_DIRECT_CANDIDATES=[]
for scale in (.1,.25,.5,1.0):
    d=(scale*proposal).clamp(-WSP_EDIT_BOUND,WSP_EDIT_BOUND)
    score=wsp_mse(d,val_x,val_y)
    WSP_DIRECT_CANDIDATES.append({'scale':scale,'delta':d.detach().cpu().tolist(),
                                   'validation_mse':score})
best=min(WSP_DIRECT_CANDIDATES,key=lambda x:x['validation_mse'])
WSP_DIRECT_ACCEPT=best['validation_mse']<.995*WSP_BASE_VAL
WSP_DIRECT_DELTA=torch.tensor(best['delta'],device=device) if WSP_DIRECT_ACCEPT else WSP_ZERO
WSP_DIRECT_S=time.perf_counter()-wsp_start
print('Numerical repair accepted:',WSP_DIRECT_ACCEPT,'val MSE:',best['validation_mse'])


In [ ]:
wsp_start=time.perf_counter()
wsp_adam_delta=nn.Parameter(torch.zeros(8,device=device))
wsp_optimizer=torch.optim.Adam([wsp_adam_delta],lr=.025)
WSP_ADAM_STEPS=40
WSP_ADAM_CANDIDATES=[]
for step in range(WSP_ADAM_STEPS):
    wsp_optimizer.zero_grad()
    ix=torch.randint(len(train_x),(min(8,len(train_x)),))
    pred=wsp_model_prediction(wsp_adam_delta,train_x[ix].to(device))
    loss=(pred-train_y[ix].to(device)).square().mean()
    loss.backward()
    wsp_optimizer.step()
    with torch.no_grad():wsp_adam_delta.clamp_(-WSP_EDIT_BOUND,WSP_EDIT_BOUND)
    if (step+1)%10==0:
        score=wsp_mse(wsp_adam_delta.detach(),val_x,val_y)
        WSP_ADAM_CANDIDATES.append({'step':step+1,
             'delta':wsp_adam_delta.detach().cpu().tolist(),'validation_mse':score})
best_adam=min(WSP_ADAM_CANDIDATES,key=lambda x:x['validation_mse'])
WSP_ADAM_ACCEPT=best_adam['validation_mse']<.995*WSP_BASE_VAL
WSP_ADAM_DELTA=torch.tensor(best_adam['delta'],device=device) if WSP_ADAM_ACCEPT else WSP_ZERO
WSP_ADAM_S=time.perf_counter()-wsp_start
print('Adam accepted:',WSP_ADAM_ACCEPT,'best validation MSE:',best_adam['validation_mse'])


## Step 7 — Benchmark **three** Hugging Face Weight-Space Programmers

We use three independently loaded open-weight instruction models:

1. `Qwen/Qwen2.5-0.5B-Instruct`
2. `Qwen/Qwen2.5-1.5B-Instruct`
3. `HuggingFaceTB/SmolLM2-1.7B-Instruct`

Each model sees the same frozen kernel values and **training-only** Jacobian statistics. Each receives **three constrained JSON proposal attempts**; only well-formed, finite eight-number corrections in ±0.2 may be evaluated. The verifier scores validation recordings and rejects proposals failing a fixed 0.5% improvement threshold. We save **all** candidates, invalid outputs and wall-clock costs. Models are loaded **sequentially** to fit a Colab T4. Ground-truth weights and held-out test labels never appear in prompts.

This is a genuine experiment in **LLM-generated neural weight modification**. The IBM TTM model belongs to the **separate time-series forecasting benchmark**, not this weight-editing experiment; TTM cannot be substituted for an instruction model that generates JSON edits.

In [ ]:
%pip -q install 'transformers>=4.45,<5' 'accelerate>=0.33' safetensors

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import re, gc
WSP_HF_MODELS=[
 'Qwen/Qwen2.5-0.5B-Instruct',
 'Qwen/Qwen2.5-1.5B-Instruct',
 'HuggingFaceTB/SmolLM2-1.7B-Instruct',
]
WSP_HF_ATTEMPTS=3
WSP_HF_MODEL_RESULTS={}

def llm_weight_proposal(model,tokenizer,attempt):
    diagnostics={
      'layer':'local.2.weight','edit_count':8,'bound':WSP_EDIT_BOUND,
      'frozen_selected_weights':np.round(WSP_FROZEN_VALUES.cpu().numpy(),5).tolist(),
      'training_JtJ':np.round(gram.detach().cpu().numpy(),5).tolist(),
      'training_Jtr':np.round(cross.detach().cpu().numpy(),5).tolist(),
      'ridge_lambda':WSP_L2,'attempt':attempt+1}
    messages=[
      {'role':'system','content':'You are a neural weight-space programmer. Return exactly JSON '
       '{"delta_weights":[eight numbers]}. Every number must be finite and within [-0.2,0.2]. '
       'You are proposing weight edits to a frozen convolution kernel. No additional text.'},
      {'role':'user','content':json.dumps(diagnostics)}]
    text=tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
    dev=next(model.parameters()).device
    inputs=tokenizer(text,return_tensors='pt').to(dev)
    with torch.inference_mode():
        output=model.generate(**inputs,max_new_tokens=180,do_sample=False,
                              pad_token_id=tokenizer.eos_token_id)
    raw=tokenizer.decode(output[0,inputs['input_ids'].shape[1]:],skip_special_tokens=True)
    for block in re.findall(r'\{[^{}]*\}',raw,re.S):
        try:
            payload=json.loads(block)
            if set(payload)!={'delta_weights'}:continue
            vals=payload['delta_weights']
            if not isinstance(vals,list) or len(vals)!=8:continue
            if any(type(v) not in (int,float) or not math.isfinite(v) or abs(v)>WSP_EDIT_BOUND for v in vals):continue
            return torch.tensor(vals,dtype=torch.float32,device=device),raw
        except (ValueError,TypeError):pass
    raise ValueError('Invalid bounded JSON: '+raw[:150])

for model_id in WSP_HF_MODELS:
    start=time.perf_counter()
    attempts=[]
    model=None;tokenizer=None
    try:
        tokenizer=AutoTokenizer.from_pretrained(model_id,token=False)
        dtype=torch.float16 if torch.cuda.is_available() else torch.float32
        model=AutoModelForCausalLM.from_pretrained(model_id,torch_dtype=dtype,token=False).to(device).eval()
        for attempt in range(WSP_HF_ATTEMPTS):
            try:
                delta,raw=llm_weight_proposal(model,tokenizer,attempt)
                validation=wsp_mse(delta,val_x,val_y)
                attempts.append({'attempt':attempt,'valid':True,'delta':delta.cpu().tolist(),
                                 'validation_mse':validation,'raw':raw[:300]})
            except Exception as exc:
                attempts.append({'attempt':attempt,'valid':False,'error':str(exc)[:250]})
        valid=[a for a in attempts if a['valid']]
        best=min(valid,key=lambda x:x['validation_mse']) if valid else None
        accepted=bool(best and best['validation_mse']<.995*WSP_BASE_VAL)
        chosen=torch.tensor(best['delta'],dtype=torch.float32,device=device) if accepted else WSP_ZERO
        status='evaluated' if valid else 'invalid_proposals'
    except Exception as exc:
        status='unavailable';accepted=False;chosen=WSP_ZERO
        attempts.append({'error':type(exc).__name__+': '+str(exc)[:200]})
    finally:
        del model,tokenizer
        gc.collect()
        if torch.cuda.is_available():torch.cuda.empty_cache()
    WSP_HF_MODEL_RESULTS[model_id]={
      'status':status,'accepted':accepted,'delta':chosen.cpu().tolist(),
      'elapsed_s':time.perf_counter()-start,'proposals':attempts}
    print(model_id,status,'accepted:',accepted,'valid proposals:',sum(a.get('valid',False) for a in attempts))

## 5. Experimental evaluation — frozen, numerical, Adam, and three independent HF programmers

We compute test outcomes **only after** validation has chosen (or rejected) the weight edit. For each method, report held-out spectral MSE, high-frequency MSE, onset proxy F1, number of changed weights, valid proposal rate and repair cost. The graph clearly emphasizes the three **proposed HF-WSP arms** with thick outlines; the existing `pcwsr_jacobian` and `pcwsr_adam` are not new methods.

In [ ]:
WSP_COMPARISON=[]
for name,delta,accepted,t in (
    ('frozen',WSP_ZERO,False,0.),
    ('pcwsr_jacobian',WSP_DIRECT_DELTA,WSP_DIRECT_ACCEPT,WSP_DIRECT_S),
    ('pcwsr_adam',WSP_ADAM_DELTA,WSP_ADAM_ACCEPT,WSP_ADAM_S)):
    m=wsp_test_metrics(delta) if 'wsp_test_metrics' in globals() else None
    if m is None:
        with torch.no_grad():
            p=torch.cat([wsp_model_prediction(delta,b.to(device)).cpu() for b in test_x.split(BENCH_BATCH)])
            m=bench_evaluate(p,test_y,test_x,WSP_THRESHOLD)
    WSP_COMPARISON.append({'model':name,'group':'repair baseline','accepted':accepted,
                           'edited_weights':int((delta.abs()>1e-7).sum()),
                           'test_mse':m['mse'],'high_frequency_mse':m['hf_mse'],
                           'onset_f1':m['onset_f1'],'time_s':t})
for model_id,out in WSP_HF_MODEL_RESULTS.items():
    delta=torch.tensor(out['delta'],device=device)
    with torch.no_grad():
        p=torch.cat([wsp_model_prediction(delta,b.to(device)).cpu() for b in test_x.split(BENCH_BATCH)])
        m=bench_evaluate(p,test_y,test_x,WSP_THRESHOLD)
    WSP_COMPARISON.append({'model':model_id,'group':'PROPOSED HF-WSP',
                           'status':out['status'],'accepted':out['accepted'],
                           'edited_weights':int((delta.abs()>1e-7).sum()),
                           'test_mse':m['mse'],'high_frequency_mse':m['hf_mse'],
                           'onset_f1':m['onset_f1'],'time_s':out['elapsed_s']})
WSP_COMPARISON_DF=pd.DataFrame(WSP_COMPARISON)
print(WSP_COMPARISON_DF.to_string(index=False))
WSP_COMPARISON_DF.to_csv('music_wsp_three_hf_proposers.csv',index=False)
Path('music_wsp_three_hf_proposals.json').write_text(json.dumps(WSP_HF_MODEL_RESULTS,indent=2))
fig,ax=plt.subplots(figsize=(12,4.5))
bars=ax.bar(WSP_COMPARISON_DF.model,WSP_COMPARISON_DF.test_mse,alpha=.64)
for b,(_,r) in zip(bars,WSP_COMPARISON_DF.iterrows()):
    if r['group']=='PROPOSED HF-WSP':b.set_hatch('///');b.set_edgecolor('black');b.set_linewidth(3)
ax.set(title='PROPOSED: three HF weight-space programmers vs numerical/Adam repair',
       ylabel='Held-out test MSE ↓')
ax.tick_params(axis='x',rotation=34)
fig.tight_layout();fig.savefig('music_wsp_three_hf_proposal_comparison.png',dpi=160);plt.show()
print('Proposed HF models evaluated:',sum(x['status']=='evaluated' for x in WSP_HF_MODEL_RESULTS.values()),'/3')

In [ ]:
# Evidence-based automatic conclusion (does not claim success for rejected edits).
frozen=float(WSP_COMPARISON_DF.loc[WSP_COMPARISON_DF.model=='frozen','test_mse'].iloc[0])
print('EVIDENCE-BASED CONCLUSION')
for _,row in WSP_COMPARISON_DF[WSP_COMPARISON_DF.group=='PROPOSED HF-WSP'].iterrows():
    if row.get('status')!='evaluated':
        print(row['model'],': unavailable or invalid; no performance claim.')
    elif not bool(row['accepted']):
        print(row['model'],': rejected on validation; test score is frozen fallback, not a successful repair.')
    else:
        improvement=100*(frozen-float(row['test_mse']))/max(frozen,1e-12)
        print(row['model'],f': accepted, held-out MSE change versus frozen = {improvement:.2f}% improvement.')
print('Future research: larger edit surfaces, real recordings, 10+ seeds, matched compute budgets.')


## Research agenda: evidentiary standards

The research questions above are intentionally harder than the experiments currently executed by this notebook. **A hypothesis is not a measured result**, and an unimplemented extension is not a contribution. The diagrams identify the proposed scientific mechanism; they must not visually encode superiority that the numerical evidence has not established.

A claim of superiority requires a preregistered outcome, independent train/validation/test groups, repeated random seeds, uncertainty estimates, robust baselines, and numerical/compute fairness. When comparing pretrained Hugging Face models, distinguish time-series forecasting checkpoints from instruction models proposing weight edits. Unavailable checkpoints and rejected repairs remain explicitly missing/rejected, never assigned fabricated performance.

The highest-impact unresolved questions are (i) separating smooth dynamics from abrupt musical events, (ii) identifying transferable local/global representations, (iii) preserving harmonic and phase detail, (iv) maintaining long-range structure, and (v) performing non-regressive, localized neural weight-space repair under distribution shift. The present notebook provides an initial component of that program and specifies which claims remain future work.


## 7. Discussion and limitations

**Discussion.** Direct weight editing is scientifically distinguishable from gradient-based training only when the generated values are actually applied to neural parameters and assessed independently. Here, an open-weight language model proposes eight convolution-kernel deltas; it neither selects the editable coordinates nor has authority to approve its own edits. The Jacobian diagnostics expose enough local optimization structure for a conventional numerical baseline, making any genuine advantage demanding to demonstrate. The model-loading overhead and invalid proposal rates must be included. Synthetic recordings and single-seed training limit external validity.

## 8. Conclusion and future research

**Conclusion.** The notebook implements a falsifiable *constrained* WSP experiment for a frozen continuous-time music network. Any empirical advantage must be established with held-out results and strong numerical and gradient controls. A rejected edit or an unavailable checkpoint is a negative or missing measurement, never an implicit successful repair.

**Future research.** Learn the repair-mask $S$ rather than selecting it manually; extend edits from eight weights to structured convolution blocks and latent dynamics; compare Qwen/SmolLM with stronger numerical and learned optimizers under matched inference budgets; evaluate diverse real recordings and at least 10 seeds; study sensitivity to injected failure modes, solver tolerances and quantization; and investigate JAX/Equinox/Diffrax for compiled Jacobian and Hessian-vector calculations.

## References

[1] M. Habiba and B. A. Pearlmutter, *Continuous Convolutional Neural Networks: Coupled Neural PDE and ODE* (2021). https://arxiv.org/abs/2111.00343

[2] R. Takehi et al., *Sparks of Weight-Space Programming* (2026). https://rikiyat.github.io/blog/weight-space-programming/

[3] R. T. Q. Chen et al., *Neural Ordinary Differential Equations*, NeurIPS (2018). https://arxiv.org/abs/1806.07366

[4] R. T. Q. Chen, `torchdiffeq`. https://github.com/rtqichen/torchdiffeq

[5] Qwen, *Qwen2.5-0.5B-Instruct* and *Qwen2.5-1.5B-Instruct*, model cards. https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct ; https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct

[6] Hugging Face, *SmolLM2-1.7B-Instruct*, model card. https://huggingface.co/HuggingFaceTB/SmolLM2-1.7B-Instruct

**Reproducibility and ethics.** Three open-weight checkpoints are proposals, not trusted scientific oracles. The notebook must record every generated candidate, rejection reason, seed, and validation outcome. Uploaded recordings must be lawful to process. No WSP efficacy is asserted until results are measured.
[7] P. Kidger et al., *Neural Controlled Differential Equations for Irregular Time Series* (2020). https://papers.neurips.cc/paper/2020/hash/4a5876b450b45371f6cfe5047ac8cd45-Abstract.html

[8] C. Hawthorne et al., *Enabling Factorized Piano Music Modeling and Generation with the MAESTRO Dataset* (2019). https://arxiv.org/abs/1810.12247

[9] P. Kidger et al., *Diffrax: Differential Equation Solvers in JAX*, solver documentation. https://docs.kidger.site/diffrax/
